In [3]:
import gffutils, os
import pandas as pd

### ok, we need a very-very fast package to work with gff files

In [4]:
# !wget https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_48/gencode.v48.annotation.gtf.gz -O ../../data/gencode.v48.annotation.gtf.gz

In [5]:
# !gunzip ../../data/gencode.v48.annotation.gtf.gz

In [6]:
# this takes ~8 min at first run

genome_dir = "../../data/annotation"
base_name = "gencode.v48.annotation.gtf"
gff_path = os.path.join(genome_dir, base_name)
db_path = os.path.join(genome_dir, f"{base_name}.db")

if not os.path.exists(db_path):
	db = gffutils.create_db(
		gff_path,
		dbfn=db_path,
		disable_infer_transcripts=True,
		disable_infer_genes=True
	)
else:
	db = gffutils.FeatureDB(db_path)

In [7]:
featuretypes = db.featuretypes()
print (list(featuretypes))

['CDS', 'Selenocysteine', 'UTR', 'exon', 'gene', 'start_codon', 'stop_codon', 'transcript']


In [8]:
%time
features = list(db.region(region=("chr1", 10_000_000, 10_400_000)))
print (len(features))


CPU times: user 2 μs, sys: 0 ns, total: 2 μs
Wall time: 4.29 μs
1505


In [9]:
# get all transcripts in a region
features = list(db.region(region=("chr1", 10_000_000, 10_400_000), featuretype="transcript"))
print (len(features))


54


In [16]:
_ = [f for f in db.all_features(limit="chr1:0-10000000",featuretype="transcript") if "ENST00000476083.1" in f.attributes["transcript_id"]]
print (_[0])

chr1	HAVANA	transcript	8945868	8974874	.	+	.	gene_id "ENSG00000131686.15"; transcript_id "ENST00000476083.1"; gene_type "protein_coding"; gene_name "CA6"; transcript_type "protein_coding_CDS_not_defined"; transcript_name "CA6-204"; level "2"; transcript_support_level "3"; hgnc_id "HGNC:1380"; havana_gene "OTTHUMG00000001763.8"; havana_transcript "OTTHUMT00000004913.1";


In [13]:
list(db.children(features[6]))

[<Feature transcript (chr1:10054445-10054781[-]) at 0x7fd7e9e57940>,
 <Feature exon (chr1:10054445-10054781[-]) at 0x7fd7e9e57310>]

In [ ]:
features[6].attributes.keys()

dict_keys(['gene_id', 'transcript_id', 'gene_type', 'gene_name', 'transcript_type', 'transcript_name', 'level', 'transcript_support_level', 'hgnc_id', 'ont', 'tag', 'havana_gene', 'havana_transcript'])

In [19]:
features[6].attributes["gene_type"], features[6].attributes["tag"], features[6].start, features[6].end, features[6].strand

(['processed_pseudogene'],
 ['pseudo_consens', 'basic', 'Ensembl_canonical', 'GENCODE_Primary'],
 10054445,
 10054781,
 '-')

In [29]:
print ("\n".join(str(features[1]).split("\t")))

chr1
HAVANA
transcript
9997228
10016021
.
+
.
gene_id "ENSG00000162444.12"; transcript_id "ENST00000294435.8"; gene_type "protein_coding"; gene_name "RBP7"; transcript_type "protein_coding"; transcript_name "RBP7-201"; level "2"; protein_id "ENSP00000294435.7"; transcript_support_level "1"; hgnc_id "HGNC:30316"; tag "basic,Ensembl_canonical,GENCODE_Primary,MANE_Select,appris_principal_1,CCDS"; ccdsid "CCDS109.1"; havana_gene "OTTHUMG00000001798.3"; havana_transcript "OTTHUMT00000005027.3";


In [29]:
all_attributes = {}
# get all possible attribute keys
num_processed = 0
for feature in db.all_features(featuretype="transcript"):
	attributes = feature.attributes
	for key, value in attributes.items():
		if type(value) == str:
			value = value.split(",")
		if key in all_attributes:
			all_attributes[key] += value
		else:
			all_attributes[key] = value
	num_processed += 1
	if num_processed > 10000:
		break

for key, value in all_attributes.items():
	if len(set(value)) > 20:
		print (key, len(set(value)), "values, examples:", sorted(list(set(value)))[:3])
	else:
		print (key, "\n", pd.Series(value).value_counts(),"\n----")	

gene_id 1915 values, examples: ['ENSG00000000938.13', 'ENSG00000001460.18', 'ENSG00000001461.19']
transcript_id 10001 values, examples: ['ENST00000003583.12', 'ENST00000003912.9', 'ENST00000010299.10']
gene_type 
 protein_coding                        5184
lncRNA                                4332
processed_pseudogene                   165
snRNA                                   61
miRNA                                   56
misc_RNA                                54
unprocessed_pseudogene                  38
transcribed_unprocessed_pseudogene      33
transcribed_processed_pseudogene        24
snoRNA                                  20
TEC                                     12
rRNA_pseudogene                          8
scaRNA                                   7
transcribed_unitary_pseudogene           5
unitary_pseudogene                       2
Name: count, dtype: int64 
----
gene_name 1876 values, examples: ['A3GALT2', 'AADACL3', 'AADACL4']
transcript_type 
 lncRNA                  

In [20]:
"sdfs".split(",")

['sdfs']

### tokenizer experiments

In [33]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("AIRI-Institute/gena-lm-bert-base-t2t")

/disk/10tb/home/fishman/miniconda3/envs/annotation/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [34]:
encoding = tok(
			"ATGAAATAGATGATGACCAGATGANATANNNNNNNNNNNNNNNNNNNNNNNNATA",
			return_tensors='pt',
			return_offsets_mapping=True,
			add_special_tokens=True,
			truncation=True,
			max_length=20,
			padding='max_length'
		)


In [35]:
for k, v in encoding.items():
	print (k,":",v)


input_ids : tensor([[    1,    28,   728,   432, 11113,  3777,  1683,     5,  1683,     2,
             3,     3,     3,     3,     3,     3,     3,     3,     3,     3]])
token_type_ids : tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
attention_mask : tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
offset_mapping : tensor([[[ 0,  0],
         [ 0,  3],
         [ 3,  9],
         [ 9, 15],
         [15, 23],
         [23, 25],
         [25, 28],
         [51, 52],
         [52, 55],
         [ 0,  0],
         [ 0,  0],
         [ 0,  0],
         [ 0,  0],
         [ 0,  0],
         [ 0,  0],
         [ 0,  0],
         [ 0,  0],
         [ 0,  0],
         [ 0,  0],
         [ 0,  0]]])


In [36]:
for om,t in zip(encoding['offset_mapping'][0], encoding['input_ids'][0]):
	print (t.item(),om.numpy(),tok.decode(t.item()))

1 [0 0] [CLS]
28 [0 3] ATG
728 [3 9] AAATAG
432 [ 9 15] ATGATG
11113 [15 23] ACCAGATG
3777 [23 25] AN
1683 [25 28] ATA
5 [51 52] -
1683 [52 55] ATA
2 [0 0] [SEP]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]


In [37]:
import re
for i in range(15):
	s = "N"*i + "ATG" + "N"*(i+1)
	# find each occurrence of 10 or more "Ns"
	matches = re.finditer(r"N{10,}", s)
	for m in matches:
		print (m.start(), m.end())
	encoding = tok(s, return_tensors='pt', return_offsets_mapping=True, add_special_tokens=True, truncation=True, max_length=20, padding='max_length')
	for om,t in zip(encoding['offset_mapping'][0], encoding['input_ids'][0]):
		print (t.item(),om.numpy(),tok.decode(t.item()))
	print (s, tok.encode(s))

1 [0 0] [CLS]
28 [0 3] ATG
12 [3 4] N
2 [0 0] [SEP]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
ATGN [1, 28, 12, 2]
1 [0 0] [CLS]
12 [0 1] N
28 [1 4] ATG
11642 [4 6] NN
2 [0 0] [SEP]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
NATGNN [1, 12, 28, 11642, 2]
1 [0 0] [CLS]
11642 [0 2] NN
28 [2 5] ATG
11642 [5 7] NN
12 [7 8] N
2 [0 0] [SEP]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
NNATGNNN [1, 11642, 28, 11642, 12, 2]
1 [0 0] [CLS]
11642 [0 2] NN
12 [2 3] N
28 [3 6] ATG
11642 [6 8] NN
11642 [ 8 10] 

In [38]:
s="ATG"
s[3]

IndexError: string index out of range

In [39]:
tokenizer = AutoTokenizer.from_pretrained("AIRI-Institute/gena-lm-bert-base-t2t")
sequence = "N"*20 + "ATGAAATAGATGATGACCAGATGANATANNNNNNNNNNNNNNNNNNNNNNNNATA"
print (len(sequence))
encoding = tokenizer(
	sequence,
	return_tensors='pt',
	return_offsets_mapping=True,
	add_special_tokens=True,
	truncation=True,
	max_length=20,
	padding='max_length'
)

for om,t in zip(encoding['offset_mapping'][0], encoding['input_ids'][0]):
	print (t.item(),om.numpy(),tokenizer.decode(t.item()))

token_ids = encoding['input_ids'].squeeze(0)
offset_mapping = encoding['offset_mapping'].squeeze(0).tolist()
for idx, token_id in enumerate(token_ids):
	if token_id.item() == 5: # gap token
		offset_mapping[idx][0] = offset_mapping[idx-1][1]

# sanity check
for tok,om in zip(token_ids, offset_mapping):
	if tok.item() != 5 and om[0] < om[1]:
		token_chars = tokenizer.decode(tok.item())
		assert token_chars[0] == sequence[om[0]]

print ("-----------------")
for om,t in zip(offset_mapping, encoding['input_ids'][0]):
	print (t.item(),om,tokenizer.decode(t.item()))


75
1 [0 0] [CLS]
5 [19 20] -
28 [20 23] ATG
728 [23 29] AAATAG
432 [29 35] ATGATG
11113 [35 43] ACCAGATG
3777 [43 45] AN
1683 [45 48] ATA
5 [71 72] -
1683 [72 75] ATA
2 [0 0] [SEP]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
3 [0 0] [PAD]
-----------------
1 [0, 0] [CLS]
5 [0, 20] -
28 [20, 23] ATG
728 [23, 29] AAATAG
432 [29, 35] ATGATG
11113 [35, 43] ACCAGATG
3777 [43, 45] AN
1683 [45, 48] ATA
5 [48, 72] -
1683 [72, 75] ATA
2 [0, 0] [SEP]
3 [0, 0] [PAD]
3 [0, 0] [PAD]
3 [0, 0] [PAD]
3 [0, 0] [PAD]
3 [0, 0] [PAD]
3 [0, 0] [PAD]
3 [0, 0] [PAD]
3 [0, 0] [PAD]
3 [0, 0] [PAD]


In [42]:
import torch
a = torch.arange(max([i[1] for i in offset_mapping]))
a

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
        36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53,
        54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71,
        72, 73, 74])

/tmp/ipykernel_1381373/2921422560.py:1: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:306.)
  a[offset_mapping]


IndexError: too many indices for tensor of dimension 1